In [5]:
# ============================================================
# CELL 1: IMPORT LIBRARIES
# ============================================================

import os
import re
import json
import warnings
import joblib

from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import display

warnings.filterwarnings("ignore")

print("✓ Libraries imported successfully.")

✓ Libraries imported successfully.


In [6]:
# ============================================================
# CELL 2: PROJECT PATHS
# ============================================================

PROJECT_ROOT = Path(
    r"C:\Users\bimal\OneDrive\Desktop\bibhu sir project\question_similarity_project"
)

DATA_DIR = PROJECT_ROOT / "data"

PROCESSED_DIR = DATA_DIR / "processed"

MODEL_DIR = PROJECT_ROOT / "model"

FEATURE_DIR = MODEL_DIR / "feature_artifacts"

FEATURE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TRAIN_PATH = (
    PROCESSED_DIR /
    "train_processed.csv"
)

VAL_PATH = (
    PROCESSED_DIR /
    "validation_processed.csv"
)

INTERNAL_TEST_PATH = (
    PROCESSED_DIR /
    "internal_test_processed.csv"
)

print("PROJECT_ROOT:")
print(PROJECT_ROOT)

print("\nFEATURE_DIR:")
print(FEATURE_DIR)

PROJECT_ROOT:
C:\Users\bimal\OneDrive\Desktop\bibhu sir project\question_similarity_project

FEATURE_DIR:
C:\Users\bimal\OneDrive\Desktop\bibhu sir project\question_similarity_project\model\feature_artifacts


In [7]:
# ============================================================
# CELL 3: VERIFY PROCESSED FILES
# ============================================================

required_files = {
    "Training": TRAIN_PATH,
    "Validation": VAL_PATH,
    "Internal Test": INTERNAL_TEST_PATH
}

for name, path in required_files.items():

    if not path.exists():

        raise FileNotFoundError(
            f"\n{name} file was not found:\n{path}\n\n"
            "Please run 2_pre_processing.ipynb first."
        )

    print(
        f"✓ {name}: {path.name}"
    )

print(
    "\n✓ All processed datasets are available."
)

✓ Training: train_processed.csv
✓ Validation: validation_processed.csv
✓ Internal Test: internal_test_processed.csv

✓ All processed datasets are available.


In [8]:
# ============================================================
# CELL 4: LOAD PROCESSED DATA
# ============================================================

train_df = pd.read_csv(
    TRAIN_PATH
)

val_df = pd.read_csv(
    VAL_PATH
)

internal_test_df = pd.read_csv(
    INTERNAL_TEST_PATH
)

print(
    "Training shape:",
    train_df.shape
)

print(
    "Validation shape:",
    val_df.shape
)

print(
    "Internal test shape:",
    internal_test_df.shape
)

Training shape: (271524, 8)
Validation shape: (76226, 8)
Internal test shape: (55986, 8)


In [9]:
# ============================================================
# CELL 5: VERIFY REQUIRED COLUMNS
# ============================================================

required_columns = [
    "question1_clean",
    "question2_clean",
    "is_duplicate"
]

for dataset_name, dataset in [
    ("train", train_df),
    ("validation", val_df),
    ("internal_test", internal_test_df)
]:

    missing_columns = [
        column
        for column in required_columns
        if column not in dataset.columns
    ]

    if missing_columns:

        raise ValueError(
            f"{dataset_name} is missing:\n"
            f"{missing_columns}"
        )

    print(
        f"✓ {dataset_name} columns verified"
    )

✓ train columns verified
✓ validation columns verified
✓ internal_test columns verified


In [10]:
# ============================================================
# CELL 6: CREATE QUESTION VARIABLES
# ============================================================

q1_train = (
    train_df["question1_clean"]
    .fillna("")
    .astype(str)
)

q2_train = (
    train_df["question2_clean"]
    .fillna("")
    .astype(str)
)

q1_val = (
    val_df["question1_clean"]
    .fillna("")
    .astype(str)
)

q2_val = (
    val_df["question2_clean"]
    .fillna("")
    .astype(str)
)

q1_test = (
    internal_test_df["question1_clean"]
    .fillna("")
    .astype(str)
)

q2_test = (
    internal_test_df["question2_clean"]
    .fillna("")
    .astype(str)
)

print("✓ Question variables created.")

print(
    "\nTraining pairs:",
    len(q1_train)
)

print(
    "Validation pairs:",
    len(q1_val)
)

print(
    "Internal test pairs:",
    len(q1_test)
)

✓ Question variables created.

Training pairs: 271524
Validation pairs: 76226
Internal test pairs: 55986


In [11]:
# ============================================================
# CELL 7: CREATE LABEL VARIABLES
# ============================================================

y_train = (
    train_df["is_duplicate"]
    .astype(int)
    .to_numpy()
)

y_val = (
    val_df["is_duplicate"]
    .astype(int)
    .to_numpy()
)

y_test = (
    internal_test_df["is_duplicate"]
    .astype(int)
    .to_numpy()
)

print(
    "y_train:",
    y_train.shape
)

print(
    "y_val:",
    y_val.shape
)

print(
    "y_test:",
    y_test.shape
)

y_train: (271524,)
y_val: (76226,)
y_test: (55986,)


In [12]:
# ============================================================
# CELL 8: FEATURE ENGINEERING STATE CHECK
# ============================================================

required_variables = {
    "train_df": train_df,
    "val_df": val_df,
    "internal_test_df": internal_test_df,

    "q1_train": q1_train,
    "q2_train": q2_train,

    "q1_val": q1_val,
    "q2_val": q2_val,

    "q1_test": q1_test,
    "q2_test": q2_test,

    "y_train": y_train,
    "y_val": y_val,
    "y_test": y_test
}

print("=" * 60)
print("FEATURE ENGINEERING STATE CHECK")
print("=" * 60)

for name, variable in required_variables.items():

    if variable is None:

        raise RuntimeError(
            f"Variable '{name}' is missing."
        )

    print(
        f"✓ {name}"
    )

print(
    "\n✓ All required variables are available."
)

FEATURE ENGINEERING STATE CHECK
✓ train_df
✓ val_df
✓ internal_test_df
✓ q1_train
✓ q2_train
✓ q1_val
✓ q2_val
✓ q1_test
✓ q2_test
✓ y_train
✓ y_val
✓ y_test

✓ All required variables are available.


In [13]:
# ============================================================
# CELL 9: WORD TF-IDF
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer

word_tfidf = TfidfVectorizer(
    lowercase=False,
    analyzer="word",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.98,
    sublinear_tf=True,
    max_features=100000,
    strip_accents="unicode"
)

word_tfidf.fit(
    pd.concat(
        [
            q1_train,
            q2_train
        ],
        ignore_index=True
    )
)

print(
    "Word TF-IDF vocabulary size:",
    len(word_tfidf.vocabulary_)
)

Word TF-IDF vocabulary size: 100000


In [14]:
# ============================================================
# CELL 10: TRANSFORM WORD TF-IDF
# ============================================================

q1_word_train = word_tfidf.transform(
    q1_train
)

q2_word_train = word_tfidf.transform(
    q2_train
)

q1_word_val = word_tfidf.transform(
    q1_val
)

q2_word_val = word_tfidf.transform(
    q2_val
)

q1_word_test = word_tfidf.transform(
    q1_test
)

q2_word_test = word_tfidf.transform(
    q2_test
)

print(
    "Training Q1 TF-IDF shape:",
    q1_word_train.shape
)

print(
    "Validation Q1 TF-IDF shape:",
    q1_word_val.shape
)

print(
    "Test Q1 TF-IDF shape:",
    q1_word_test.shape
)

Training Q1 TF-IDF shape: (271524, 100000)
Validation Q1 TF-IDF shape: (76226, 100000)
Test Q1 TF-IDF shape: (55986, 100000)


In [15]:
# ============================================================
# CELL 11: SAVE WORD TF-IDF
# ============================================================

import joblib

WORD_TFIDF_PATH = (
    FEATURE_DIR /
    "word_tfidf.pkl"
)

joblib.dump(
    word_tfidf,
    WORD_TFIDF_PATH
)

print(
    "Saved:",
    WORD_TFIDF_PATH
)

Saved: C:\Users\bimal\OneDrive\Desktop\bibhu sir project\question_similarity_project\model\feature_artifacts\word_tfidf.pkl


In [16]:
# ============================================================
# CELL 12: CHARACTER TF-IDF
# ============================================================

char_tfidf = TfidfVectorizer(
    lowercase=False,
    analyzer="char",
    ngram_range=(3, 5),
    min_df=2,
    max_df=0.98,
    sublinear_tf=True,
    max_features=100000
)

char_tfidf.fit(
    pd.concat(
        [
            q1_train,
            q2_train
        ],
        ignore_index=True
    )
)

print(
    "Character TF-IDF vocabulary size:",
    len(char_tfidf.vocabulary_)
)

Character TF-IDF vocabulary size: 100000


In [17]:
# ============================================================
# CELL 13: TRANSFORM CHARACTER TF-IDF
# ============================================================

q1_char_train = char_tfidf.transform(
    q1_train
)

q2_char_train = char_tfidf.transform(
    q2_train
)

q1_char_val = char_tfidf.transform(
    q1_val
)

q2_char_val = char_tfidf.transform(
    q2_val
)

q1_char_test = char_tfidf.transform(
    q1_test
)

q2_char_test = char_tfidf.transform(
    q2_test
)

print(
    "Character TF-IDF training shape:",
    q1_char_train.shape
)

print(
    "Character TF-IDF validation shape:",
    q1_char_val.shape
)

print(
    "Character TF-IDF test shape:",
    q1_char_test.shape
)

Character TF-IDF training shape: (271524, 100000)
Character TF-IDF validation shape: (76226, 100000)
Character TF-IDF test shape: (55986, 100000)


In [18]:
# ============================================================
# CELL 14: SAVE CHARACTER TF-IDF
# ============================================================

CHAR_TFIDF_PATH = (
    FEATURE_DIR /
    "char_tfidf.pkl"
)

joblib.dump(
    char_tfidf,
    CHAR_TFIDF_PATH
)

print(
    "Saved:",
    CHAR_TFIDF_PATH
)

Saved: C:\Users\bimal\OneDrive\Desktop\bibhu sir project\question_similarity_project\model\feature_artifacts\char_tfidf.pkl


In [19]:
# ============================================================
# CELL 15: TOKENIZATION
# ============================================================

TOKEN_PATTERN = re.compile(
    r"\b\w+\b",
    re.UNICODE
)


def tokenize(text):

    return TOKEN_PATTERN.findall(
        str(text).lower()
    )


print(
    "Tokenizer created."
)

Tokenizer created.


In [20]:
# ============================================================
# CELL 16: BASIC PAIR FEATURES
# ============================================================

def basic_pair_features(q1, q2):

    tokens1 = tokenize(q1)
    tokens2 = tokenize(q2)

    set1 = set(tokens1)
    set2 = set(tokens2)

    len_chars_1 = len(q1)
    len_chars_2 = len(q2)

    len_words_1 = len(tokens1)
    len_words_2 = len(tokens2)

    common_tokens = set1 & set2
    union_tokens = set1 | set2

    # Jaccard similarity
    if len(union_tokens) > 0:

        jaccard = (
            len(common_tokens)
            /
            len(union_tokens)
        )

    else:

        jaccard = 0.0

    # Overlap coefficient
    min_set_size = min(
        len(set1),
        len(set2)
    )

    if min_set_size > 0:

        overlap_coefficient = (
            len(common_tokens)
            /
            min_set_size
        )

    else:

        overlap_coefficient = 0.0

    # Common word ratio
    max_set_size = max(
        len(set1),
        len(set2)
    )

    if max_set_size > 0:

        common_word_ratio = (
            len(common_tokens)
            /
            max_set_size
        )

    else:

        common_word_ratio = 0.0

    # Length differences
    char_difference = abs(
        len_chars_1 -
        len_chars_2
    )

    word_difference = abs(
        len_words_1 -
        len_words_2
    )

    # Ratios
    char_ratio = (
        min(len_chars_1, len_chars_2)
        /
        max(len_chars_1, len_chars_2)
        if max(len_chars_1, len_chars_2) > 0
        else 0.0
    )

    word_ratio = (
        min(len_words_1, len_words_2)
        /
        max(len_words_1, len_words_2)
        if max(len_words_1, len_words_2) > 0
        else 0.0
    )

    # Exact normalized match
    exact_match = (
        int(q1.strip() == q2.strip())
    )

    return [
        len_chars_1,
        len_chars_2,
        len_words_1,
        len_words_2,
        char_difference,
        word_difference,
        char_ratio,
        word_ratio,
        len(common_tokens),
        jaccard,
        overlap_coefficient,
        common_word_ratio,
        exact_match
    ]


HANDCRAFTED_FEATURE_NAMES = [
    "q1_char_count",
    "q2_char_count",
    "q1_word_count",
    "q2_word_count",
    "char_length_difference",
    "word_length_difference",
    "char_length_ratio",
    "word_length_ratio",
    "common_token_count",
    "jaccard_similarity",
    "overlap_coefficient",
    "common_word_ratio",
    "exact_match"
]

print(
    "Number of handcrafted features:",
    len(HANDCRAFTED_FEATURE_NAMES)
)

Number of handcrafted features: 13


In [21]:
# ============================================================
# CELL 17: GENERATE HANDCRAFTED FEATURES
# ============================================================

def build_handcrafted_features(
    q1_series,
    q2_series
):

    features = [
        basic_pair_features(
            q1,
            q2
        )
        for q1, q2 in zip(
            q1_series,
            q2_series
        )
    ]

    return np.asarray(
        features,
        dtype=np.float32
    )


hand_train = build_handcrafted_features(
    q1_train,
    q2_train
)

hand_val = build_handcrafted_features(
    q1_val,
    q2_val
)

hand_test = build_handcrafted_features(
    q1_test,
    q2_test
)

print(
    "Training:",
    hand_train.shape
)

print(
    "Validation:",
    hand_val.shape
)

print(
    "Internal test:",
    hand_test.shape
)

Training: (271524, 13)
Validation: (76226, 13)
Internal test: (55986, 13)


In [22]:
# ============================================================
# CELL 18: INSPECT HANDCRAFTED FEATURES
# ============================================================

hand_train_df = pd.DataFrame(
    hand_train,
    columns=HANDCRAFTED_FEATURE_NAMES
)

display(
    hand_train_df.head(10)
)

,q1_char_count,q2_char_count,q1_word_count,q2_word_count,char_length_difference,word_length_difference,char_length_ratio,word_length_ratio,common_token_count,jaccard_similarity,overlap_coefficient,common_word_ratio,exact_match
0,51.0,88.0,10.0,15.0,37.0,5.0,0.579545,0.666667,7.0,0.411765,0.700000,0.500000,0.0
1,73.0,59.0,14.0,10.0,14.0,4.0,0.808219,0.714286,4.0,0.200000,0.400000,0.285714,0.0
2,50.0,65.0,11.0,13.0,15.0,2.0,0.769231,0.846154,0.0,0.000000,0.000000,0.000000,0.0
3,76.0,39.0,13.0,7.0,37.0,6.0,0.513158,0.538462,4.0,0.250000,0.571429,0.307692,0.0
4,86.0,90.0,17.0,17.0,4.0,0.0,0.955556,1.000000,11.0,0.523810,0.687500,0.687500,0.0
5,19.0,62.0,4.0,11.0,43.0,7.0,0.306452,0.363636,0.0,0.000000,0.000000,0.000000,0.0
6,60.0,49.0,9.0,9.0,11.0,0.0,0.816667,1.000000,4.0,0.285714,0.444444,0.444444,0.0
7,57.0,116.0,9.0,19.0,59.0,10.0,0.491379,0.473684,1.0,0.041667,0.111111,0.062500,0.0
8,141.0,140.0,29.0,29.0,1.0,0.0,0.992908,1.000000,21.0,0.913043,0.954545,0.954545,0.0
9,93.0,91.0,16.0,17.0,2.0,1.0,0.978495,0.941176,4.0,0.148148,0.266667,0.250000,0.0


In [23]:
# ============================================================
# CELL 19: HANDCRAFTED FEATURE STATISTICS
# ============================================================

display(
    hand_train_df.describe().T
)

,count,mean,std,min,25%,50%,75%,max
q1_char_count,271524.0,59.845718,30.046843,1.0,39.000000,52.000000,73.000000,623.0
q2_char_count,271524.0,60.396210,33.503632,0.0,39.000000,51.000000,72.000000,1169.0
q1_word_count,271524.0,11.152808,5.589339,0.0,7.000000,10.000000,13.000000,128.0
q2_word_count,271524.0,11.397147,6.380023,0.0,7.000000,10.000000,13.000000,232.0
char_length_difference,271524.0,20.244299,25.128048,0.0,5.000000,12.000000,26.000000,1038.0
word_length_difference,271524.0,3.782502,4.839426,0.0,1.000000,2.000000,5.000000,204.0
char_length_ratio,271524.0,0.753005,0.188403,0.0,0.630769,0.789474,0.911111,1.0
word_length_ratio,271524.0,0.757335,0.193783,0.0,0.625000,0.800000,0.900000,1.0
common_token_count,271524.0,5.034163,3.144896,0.0,3.000000,5.000000,7.000000,40.0
jaccard_similarity,271524.0,0.369249,0.243245,0.0,0.178571,0.333333,0.538462,1.0


In [24]:
# ============================================================
# CELL 20: MEMORY-EFFICIENT TF-IDF COSINE SIMILARITY
# ============================================================

import numpy as np


def pairwise_sparse_cosine_similarity(
    matrix1,
    matrix2
):
    """
    Calculate cosine similarity only between
    corresponding rows:

        matrix1[i] <-> matrix2[i]

    This avoids constructing the full
    N x N similarity matrix.
    """

    # Row-wise dot product
    dot_product = matrix1.multiply(matrix2).sum(
        axis=1
    )

    dot_product = np.asarray(
        dot_product
    ).ravel()

    # Row-wise L2 norms
    norm1 = np.sqrt(
        matrix1.multiply(matrix1).sum(
            axis=1
        )
    )

    norm2 = np.sqrt(
        matrix2.multiply(matrix2).sum(
            axis=1
        )
    )

    norm1 = np.asarray(
        norm1
    ).ravel()

    norm2 = np.asarray(
        norm2
    ).ravel()

    denominator = norm1 * norm2

    # Avoid division by zero
    similarity = np.divide(
        dot_product,
        denominator,
        out=np.zeros_like(
            dot_product,
            dtype=np.float64
        ),
        where=denominator != 0
    )

    return similarity.astype(
        np.float32
    )


# ------------------------------------------------------------
# WORD TF-IDF SIMILARITY
# ------------------------------------------------------------

word_cos_train = pairwise_sparse_cosine_similarity(
    q1_word_train,
    q2_word_train
)

word_cos_val = pairwise_sparse_cosine_similarity(
    q1_word_val,
    q2_word_val
)

word_cos_test = pairwise_sparse_cosine_similarity(
    q1_word_test,
    q2_word_test
)


# ------------------------------------------------------------
# CHARACTER TF-IDF SIMILARITY
# ------------------------------------------------------------

char_cos_train = pairwise_sparse_cosine_similarity(
    q1_char_train,
    q2_char_train
)

char_cos_val = pairwise_sparse_cosine_similarity(
    q1_char_val,
    q2_char_val
)

char_cos_test = pairwise_sparse_cosine_similarity(
    q1_char_test,
    q2_char_test
)


print("=" * 65)
print("PAIRWISE TF-IDF COSINE SIMILARITY")
print("=" * 65)

print(
    "\nWord cosine:"
)

print(
    "  Train:",
    word_cos_train.shape
)

print(
    "  Validation:",
    word_cos_val.shape
)

print(
    "  Test:",
    word_cos_test.shape
)

print(
    "\nCharacter cosine:"
)

print(
    "  Train:",
    char_cos_train.shape
)

print(
    "  Validation:",
    char_cos_val.shape
)

print(
    "  Test:",
    char_cos_test.shape
)

print(
    "\n✓ Memory-efficient cosine similarity completed."
)

PAIRWISE TF-IDF COSINE SIMILARITY

Word cosine:
  Train: (271524,)
  Validation: (76226,)
  Test: (55986,)

Character cosine:
  Train: (271524,)
  Validation: (76226,)
  Test: (55986,)

✓ Memory-efficient cosine similarity completed.


In [25]:
# ============================================================
# CELL 21: ADD TF-IDF SIMILARITY FEATURES
# ============================================================

hand_train = np.column_stack([
    hand_train,
    word_cos_train,
    char_cos_train
]).astype(np.float32)

hand_val = np.column_stack([
    hand_val,
    word_cos_val,
    char_cos_val
]).astype(np.float32)

hand_test = np.column_stack([
    hand_test,
    word_cos_test,
    char_cos_test
]).astype(np.float32)


HANDCRAFTED_FEATURE_NAMES += [
    "word_tfidf_cosine",
    "char_tfidf_cosine"
]

print(
    "Updated handcrafted feature count:",
    len(HANDCRAFTED_FEATURE_NAMES)
)

Updated handcrafted feature count: 15


In [26]:
# ============================================================
# CELL 22: FEATURE CORRELATION WITH TARGET
# ============================================================

feature_analysis_df = pd.DataFrame(
    hand_train,
    columns=HANDCRAFTED_FEATURE_NAMES
)

feature_analysis_df["target"] = y_train

correlations = (
    feature_analysis_df
    .corr(numeric_only=True)["target"]
    .drop("target")
    .sort_values(
        ascending=False
    )
)

display(
    correlations.to_frame(
        "correlation_with_target"
    )
)

,correlation_with_target
char_tfidf_cosine,0.382181
overlap_coefficient,0.363982
common_word_ratio,0.351860
jaccard_similarity,0.332782
word_tfidf_cosine,0.261658
common_token_count,0.217430
char_length_ratio,0.185845
word_length_ratio,0.176280
exact_match,0.009649
q1_word_count,-0.147584


In [27]:
# ============================================================
# CELL 23: SAVE HANDCRAFTED FEATURE NAMES
# ============================================================

HAND_FEATURE_NAMES_PATH = (
    FEATURE_DIR /
    "handcrafted_feature_names.json"
)

with open(
    HAND_FEATURE_NAMES_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        HANDCRAFTED_FEATURE_NAMES,
        f,
        indent=4
    )

print(
    "Saved:",
    HAND_FEATURE_NAMES_PATH
)

Saved: C:\Users\bimal\OneDrive\Desktop\bibhu sir project\question_similarity_project\model\feature_artifacts\handcrafted_feature_names.json


In [28]:
import sys

print("Python executable:")
print(sys.executable)

print("\nPython version:")
print(sys.version)

Python executable:
c:\Users\bimal\anaconda3\python.exe

Python version:
3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]


In [29]:
import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "sentence-transformers"
])

0

In [30]:
# ============================================================
# CELL 24: CHECK SENTENCE TRANSFORMERS
# ============================================================

try:

    from sentence_transformers import SentenceTransformer

    print(
        "✓ sentence-transformers is available."
    )

except ImportError:

    raise ImportError(
        "\n"
        "sentence-transformers is not installed.\n\n"
        "Install it in the VS Code terminal with:\n\n"
        "pip install sentence-transformers\n"
    )

✓ sentence-transformers is available.


In [31]:
# ============================================================
# CELL 25: LOAD SEMANTIC MODEL
# ============================================================

EMBEDDING_MODEL_NAME = (
    "sentence-transformers/all-MiniLM-L6-v2"
)

print(
    "Loading:",
    EMBEDDING_MODEL_NAME
)

semantic_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

print(
    "✓ Semantic model loaded."
)

Loading: sentence-transformers/all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✓ Semantic model loaded.


In [32]:
# ============================================================
# CELL 26: BUILD UNIQUE QUESTION CORPUS
# ============================================================

all_questions = pd.concat(
    [
        q1_train,
        q2_train,
        q1_val,
        q2_val,
        q1_test,
        q2_test
    ],
    ignore_index=True
)

unique_questions = (
    all_questions
    .drop_duplicates()
    .tolist()
)

print(
    "Total question occurrences:",
    f"{len(all_questions):,}"
)

print(
    "Unique questions:",
    f"{len(unique_questions):,}"
)

Total question occurrences: 807,472
Unique questions: 537,142


In [33]:
# ============================================================
# CELL 27: GENERATE SEMANTIC EMBEDDINGS
# ============================================================

QUESTION_EMBEDDING_BATCH_SIZE = 64

unique_embeddings = semantic_model.encode(
    unique_questions,
    batch_size=QUESTION_EMBEDDING_BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

unique_embeddings = (
    unique_embeddings
    .astype(np.float32)
)

print(
    "Embedding matrix shape:",
    unique_embeddings.shape
)

Batches:   0%|          | 0/8393 [00:00<?, ?it/s]

Embedding matrix shape: (537142, 384)


In [34]:
   # ============================================================
# CELL 28: CREATE EMBEDDING LOOKUP
# ============================================================

embedding_lookup = {
    question: embedding
    for question, embedding
    in zip(
        unique_questions,
        unique_embeddings
    )
}

print(
    "Embedding lookup created."
)

print(
    "Number of entries:",
    len(embedding_lookup)
)

Embedding lookup created.
Number of entries: 537142


In [35]:
# ============================================================
# CELL 29: BUILD PAIR EMBEDDINGS
# ============================================================

def get_pair_embeddings(
    q1_series,
    q2_series,
    lookup
):

    emb1 = np.vstack([
        lookup[q]
        for q in q1_series
    ])

    emb2 = np.vstack([
        lookup[q]
        for q in q2_series
    ])

    return (
        emb1.astype(np.float32),
        emb2.astype(np.float32)
    )


emb1_train, emb2_train = get_pair_embeddings(
    q1_train,
    q2_train,
    embedding_lookup
)

emb1_val, emb2_val = get_pair_embeddings(
    q1_val,
    q2_val,
    embedding_lookup
)

emb1_test, emb2_test = get_pair_embeddings(
    q1_test,
    q2_test,
    embedding_lookup
)

print(
    "Training embedding shape:",
    emb1_train.shape
)

print(
    "Validation embedding shape:",
    emb1_val.shape
)

print(
    "Internal test embedding shape:",
    emb1_test.shape
)

Training embedding shape: (271524, 384)
Validation embedding shape: (76226, 384)
Internal test embedding shape: (55986, 384)


In [36]:
# ============================================================
# CELL 30: SEMANTIC COSINE SIMILARITY
# ============================================================

semantic_cos_train = np.sum(
    emb1_train * emb2_train,
    axis=1
)

semantic_cos_val = np.sum(
    emb1_val * emb2_val,
    axis=1
)

semantic_cos_test = np.sum(
    emb1_test * emb2_test,
    axis=1
)

print(
    "Semantic cosine statistics:"
)

print(
    pd.Series(
        semantic_cos_train
    ).describe()
)

Semantic cosine statistics:
count    271524.000000
mean          0.670522
std           0.269726
min          -0.265854
25%           0.528712
50%           0.752559
75%           0.878097
max           1.000000
dtype: float64


In [37]:
# ============================================================
# CELL 31: SEMANTIC INTERACTION FEATURES
# ============================================================

semantic_abs_train = np.abs(
    emb1_train -
    emb2_train
)

semantic_mul_train = (
    emb1_train *
    emb2_train
)

semantic_abs_val = np.abs(
    emb1_val -
    emb2_val
)

semantic_mul_val = (
    emb1_val *
    emb2_val
)

semantic_abs_test = np.abs(
    emb1_test -
    emb2_test
)

semantic_mul_test = (
    emb1_test *
    emb2_test
)

print(
    "Semantic absolute difference:",
    semantic_abs_train.shape
)

print(
    "Semantic element-wise product:",
    semantic_mul_train.shape
)

Semantic absolute difference: (271524, 384)
Semantic element-wise product: (271524, 384)


In [38]:
# ============================================================
# CELL 32: COMPACT SEMANTIC FEATURES
# ============================================================

def build_compact_semantic_features(
    emb1,
    emb2
):

    cosine = np.sum(
        emb1 * emb2,
        axis=1
    )

    euclidean = np.linalg.norm(
        emb1 - emb2,
        axis=1
    )

    manhattan = np.sum(
        np.abs(
            emb1 - emb2
        ),
        axis=1
    )

    abs_mean = np.mean(
        np.abs(
            emb1 - emb2
        ),
        axis=1
    )

    product_mean = np.mean(
        emb1 * emb2,
        axis=1
    )

    product_max = np.max(
        emb1 * emb2,
        axis=1
    )

    product_min = np.min(
        emb1 * emb2,
        axis=1
    )

    return np.column_stack([
        cosine,
        euclidean,
        manhattan,
        abs_mean,
        product_mean,
        product_max,
        product_min
    ]).astype(np.float32)


semantic_compact_train = (
    build_compact_semantic_features(
        emb1_train,
        emb2_train
    )
)

semantic_compact_val = (
    build_compact_semantic_features(
        emb1_val,
        emb2_val
    )
)

semantic_compact_test = (
    build_compact_semantic_features(
        emb1_test,
        emb2_test
    )
)

print(
    "Training semantic features:",
    semantic_compact_train.shape
)

print(
    "Validation semantic features:",
    semantic_compact_val.shape
)

print(
    "Test semantic features:",
    semantic_compact_test.shape
)

Training semantic features: (271524, 7)
Validation semantic features: (76226, 7)
Test semantic features: (55986, 7)


In [39]:
# ============================================================
# CELL 33: SEMANTIC FEATURE NAMES
# ============================================================

SEMANTIC_FEATURE_NAMES = [
    "semantic_cosine",
    "semantic_euclidean",
    "semantic_manhattan",
    "semantic_abs_mean",
    "semantic_product_mean",
    "semantic_product_max",
    "semantic_product_min"
]

print(
    "Semantic feature count:",
    len(SEMANTIC_FEATURE_NAMES)
)

Semantic feature count: 7


In [40]:
# ============================================================
# CELL 34: SEMANTIC SIMILARITY BY CLASS
# ============================================================

semantic_analysis = pd.DataFrame({
    "semantic_cosine": semantic_cos_train,
    "target": y_train
})

display(
    semantic_analysis
    .groupby("target")[
        "semantic_cosine"
    ]
    .describe()
)

,count,mean,std,min,25%,50%,75%,max
target,,,,,,,,
0,172450.0,0.558255,0.273507,-0.265854,0.394882,0.609527,0.775808,1.0
1,99074.0,0.865936,0.095177,0.282141,0.808025,0.882615,0.941141,1.0


In [41]:
# ============================================================
# CELL 35: LEXICAL SIMILARITY BY CLASS
# ============================================================

similarity_analysis = pd.DataFrame({
    "word_tfidf_cosine": word_cos_train,
    "char_tfidf_cosine": char_cos_train,
    "semantic_cosine": semantic_cos_train,
    "target": y_train
})

display(
    similarity_analysis
    .groupby("target")
    .mean()
)

,word_tfidf_cosine,char_tfidf_cosine,semantic_cosine
target,,,
0,0.327174,0.360489,0.558255
1,0.473951,0.560242,0.865936


In [42]:
# ============================================================
# CELL 36: COMBINE COMPACT FEATURES
# ============================================================

X_basic_train = hand_train

X_basic_val = hand_val

X_basic_test = hand_test


X_advanced_train = np.column_stack([
    hand_train,
    semantic_compact_train
]).astype(np.float32)

X_advanced_val = np.column_stack([
    hand_val,
    semantic_compact_val
]).astype(np.float32)

X_advanced_test = np.column_stack([
    hand_test,
    semantic_compact_test
]).astype(np.float32)


print(
    "Basic training features:",
    X_basic_train.shape
)

print(
    "Advanced training features:",
    X_advanced_train.shape
)

print(
    "Advanced validation features:",
    X_advanced_val.shape
)

print(
    "Advanced internal test features:",
    X_advanced_test.shape
)

Basic training features: (271524, 15)
Advanced training features: (271524, 22)
Advanced validation features: (76226, 22)
Advanced internal test features: (55986, 22)


In [43]:
# ============================================================
# CELL 37: ADVANCED FEATURE NAMES
# ============================================================

ADVANCED_FEATURE_NAMES = (
    HANDCRAFTED_FEATURE_NAMES
    +
    SEMANTIC_FEATURE_NAMES
)

print(
    "Total advanced features:",
    len(ADVANCED_FEATURE_NAMES)
)

Total advanced features: 22


In [44]:
# ============================================================
# CELL 38: CREATE FEATURE DATAFRAMES
# ============================================================

basic_train_df = pd.DataFrame(
    X_basic_train,
    columns=HANDCRAFTED_FEATURE_NAMES
)

basic_val_df = pd.DataFrame(
    X_basic_val,
    columns=HANDCRAFTED_FEATURE_NAMES
)

basic_test_df = pd.DataFrame(
    X_basic_test,
    columns=HANDCRAFTED_FEATURE_NAMES
)


advanced_train_df = pd.DataFrame(
    X_advanced_train,
    columns=ADVANCED_FEATURE_NAMES
)

advanced_val_df = pd.DataFrame(
    X_advanced_val,
    columns=ADVANCED_FEATURE_NAMES
)

advanced_test_df = pd.DataFrame(
    X_advanced_test,
    columns=ADVANCED_FEATURE_NAMES
)

print(
    "Feature DataFrames created."
)

Feature DataFrames created.


In [45]:
# ============================================================
# CELL 39: FEATURE QUALITY CHECK
# ============================================================

for name, matrix in [
    ("basic_train", X_basic_train),
    ("basic_val", X_basic_val),
    ("basic_test", X_basic_test),
    ("advanced_train", X_advanced_train),
    ("advanced_val", X_advanced_val),
    ("advanced_test", X_advanced_test)
]:

    nan_count = np.isnan(matrix).sum()

    inf_count = np.isinf(matrix).sum()

    print(
        f"{name:20s} | "
        f"NaN: {nan_count:6d} | "
        f"Inf: {inf_count:6d}"
    )

    if nan_count > 0 or inf_count > 0:

        raise ValueError(
            f"Invalid values found in {name}"
        )

print(
    "\n✓ All compact feature matrices are valid."
)

basic_train          | NaN:      0 | Inf:      0
basic_val            | NaN:      0 | Inf:      0
basic_test           | NaN:      0 | Inf:      0
advanced_train       | NaN:      0 | Inf:      0
advanced_val         | NaN:      0 | Inf:      0
advanced_test        | NaN:      0 | Inf:      0

✓ All compact feature matrices are valid.


In [46]:
# ============================================================
# CELL 40: FEATURE SCALING
# ============================================================

from sklearn.preprocessing import StandardScaler
import numpy as np

print("Starting feature scaling...")

# Create scaler
feature_scaler = StandardScaler()

# ------------------------------------------------------------
# Scale training features
# ------------------------------------------------------------
X_advanced_train_scaled = (
    feature_scaler.fit_transform(
        X_advanced_train
    )
    .astype(np.float32)
)

# ------------------------------------------------------------
# Scale validation features
# ------------------------------------------------------------
X_advanced_val_scaled = (
    feature_scaler.transform(
        X_advanced_val
    )
    .astype(np.float32)
)

# ------------------------------------------------------------
# Scale test features
# ------------------------------------------------------------
X_advanced_test_scaled = (
    feature_scaler.transform(
        X_advanced_test
    )
    .astype(np.float32)
)

print("✓ Feature scaling completed")

print(
    "Train shape:",
    X_advanced_train_scaled.shape
)

print(
    "Validation shape:",
    X_advanced_val_scaled.shape
)

print(
    "Test shape:",
    X_advanced_test_scaled.shape
)

Starting feature scaling...
✓ Feature scaling completed
Train shape: (271524, 22)
Validation shape: (76226, 22)
Test shape: (55986, 22)


In [47]:
# ============================================================
# CELL 41: SAVE FEATURE SCALER
# ============================================================

SCALER_PATH = (
    FEATURE_DIR /
    "feature_scaler.pkl"
)

joblib.dump(
    feature_scaler,
    SCALER_PATH
)

print(
    "Saved:",
    SCALER_PATH
)

Saved: C:\Users\bimal\OneDrive\Desktop\bibhu sir project\question_similarity_project\model\feature_artifacts\feature_scaler.pkl


In [48]:
# ============================================================
# CELL 42: SAVE COMPACT FEATURE MATRICES
# ============================================================

np.savez_compressed(
    FEATURE_DIR / "basic_features.npz",
    X_train=X_basic_train,
    X_val=X_basic_val,
    X_test=X_basic_test
)

np.savez_compressed(
    FEATURE_DIR / "advanced_features.npz",
    X_train=X_advanced_train,
    X_val=X_advanced_val,
    X_test=X_advanced_test
)

np.savez_compressed(
    FEATURE_DIR / "advanced_features_scaled.npz",
    X_train=X_advanced_train_scaled,
    X_val=X_advanced_val_scaled,
    X_test=X_advanced_test_scaled
)

print(
    "✓ Basic feature matrix saved."
)

print(
    "✓ Advanced feature matrix saved."
)

print(
    "✓ Scaled advanced feature matrix saved."
)

✓ Basic feature matrix saved.
✓ Advanced feature matrix saved.
✓ Scaled advanced feature matrix saved.


In [49]:
# ============================================================
# CELL 43: SAVE LABELS
# ============================================================

np.save(
    FEATURE_DIR / "y_train.npy",
    y_train
)

np.save(
    FEATURE_DIR / "y_val.npy",
    y_val
)

np.save(
    FEATURE_DIR / "y_test.npy",
    y_test
)

print(
    "Labels saved successfully."
)

Labels saved successfully.


In [50]:
# ============================================================
# CELL 44: SAVE SEMANTIC EMBEDDINGS
# ============================================================

np.savez_compressed(
    FEATURE_DIR / "semantic_pair_embeddings.npz",

    emb1_train=emb1_train,
    emb2_train=emb2_train,

    emb1_val=emb1_val,
    emb2_val=emb2_val,

    emb1_test=emb1_test,
    emb2_test=emb2_test
)

print(
    "Semantic pair embeddings saved."
)

Semantic pair embeddings saved.


In [51]:
# ============================================================
# CELL 45: SAVE SEMANTIC MODEL CONFIGURATION
# ============================================================

semantic_config = {
    "model_name": EMBEDDING_MODEL_NAME,
    "embedding_dimension": int(
        unique_embeddings.shape[1]
    ),
    "normalized_embeddings": True,
    "batch_size": QUESTION_EMBEDDING_BATCH_SIZE
}

SEMANTIC_CONFIG_PATH = (
    FEATURE_DIR /
    "semantic_model_config.json"
)

with open(
    SEMANTIC_CONFIG_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        semantic_config,
        f,
        indent=4
    )

print(
    "Saved:",
    SEMANTIC_CONFIG_PATH
)

Saved: C:\Users\bimal\OneDrive\Desktop\bibhu sir project\question_similarity_project\model\feature_artifacts\semantic_model_config.json


In [52]:
# ============================================================
# CELL 46: SAVE FEATURE CONFIGURATION
# ============================================================

feature_config = {
    "basic_features": HANDCRAFTED_FEATURE_NAMES,
    "semantic_features": SEMANTIC_FEATURE_NAMES,
    "advanced_features": ADVANCED_FEATURE_NAMES
}

FEATURE_CONFIG_PATH = (
    FEATURE_DIR /
    "feature_config.json"
)

with open(
    FEATURE_CONFIG_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        feature_config,
        f,
        indent=4
    )

print(
    "Saved:",
    FEATURE_CONFIG_PATH
)

Saved: C:\Users\bimal\OneDrive\Desktop\bibhu sir project\question_similarity_project\model\feature_artifacts\feature_config.json


In [53]:
# ============================================================
# CELL 47: ADVANCED FEATURE STATISTICS
# ============================================================

display(
    advanced_train_df.describe().T
)

,count,mean,std,min,25%,50%,75%,max
q1_char_count,271524.0,59.845718,30.046843,1.000000,39.000000,52.000000,73.000000,623.000000
q2_char_count,271524.0,60.396210,33.503632,0.000000,39.000000,51.000000,72.000000,1169.000000
q1_word_count,271524.0,11.152808,5.589339,0.000000,7.000000,10.000000,13.000000,128.000000
q2_word_count,271524.0,11.397147,6.380023,0.000000,7.000000,10.000000,13.000000,232.000000
char_length_difference,271524.0,20.244299,25.128048,0.000000,5.000000,12.000000,26.000000,1038.000000
word_length_difference,271524.0,3.782502,4.839426,0.000000,1.000000,2.000000,5.000000,204.000000
char_length_ratio,271524.0,0.753005,0.188403,0.000000,0.630769,0.789474,0.911111,1.000000
word_length_ratio,271524.0,0.757335,0.193783,0.000000,0.625000,0.800000,0.900000,1.000000
common_token_count,271524.0,5.034163,3.144896,0.000000,3.000000,5.000000,7.000000,40.000000
jaccard_similarity,271524.0,0.369249,0.243245,0.000000,0.178571,0.333333,0.538462,1.000000


In [54]:
# ============================================================
# CELL 48: FEATURE MEANS BY CLASS
# ============================================================

diagnostic_df = advanced_train_df.copy()

diagnostic_df["target"] = y_train

class_means = (
    diagnostic_df
    .groupby("target")
    .mean(numeric_only=True)
    .T
)

class_means.columns = [
    "different_0",
    "duplicate_1"
]

class_means["absolute_difference"] = (
    class_means["duplicate_1"]
    -
    class_means["different_0"]
).abs()

class_means = (
    class_means
    .sort_values(
        "absolute_difference",
        ascending=False
    )
)

display(
    class_means
)

,different_0,duplicate_1,absolute_difference
char_length_difference,24.213285,13.335810,10.877476
q2_char_count,64.187737,53.796608,10.391129
q1_char_count,63.391708,53.673508,9.718201
semantic_manhattan,13.845390,7.468638,6.376752
q2_word_count,12.155025,10.077972,2.077052
word_length_difference,4.531447,2.478874,2.052572
q1_word_count,11.778162,10.064305,1.713857
common_token_count,4.515871,5.936310,1.420439
semantic_euclidean,0.890877,0.480846,0.410030
semantic_cosine,0.558255,0.865936,0.307680


In [55]:
# ============================================================
# CELL 49: SAVE FEATURE DIAGNOSTICS
# ============================================================

DIAGNOSTIC_PATH = (
    FEATURE_DIR /
    "feature_class_diagnostics.csv"
)

class_means.to_csv(
    DIAGNOSTIC_PATH
)

print(
    "Saved:",
    DIAGNOSTIC_PATH
)

Saved: C:\Users\bimal\OneDrive\Desktop\bibhu sir project\question_similarity_project\model\feature_artifacts\feature_class_diagnostics.csv


In [56]:
# ============================================================
# CELL 50: VERIFY FEATURE ARTIFACTS
# ============================================================

expected_artifacts = [
    "word_tfidf.pkl",
    "char_tfidf.pkl",
    "handcrafted_feature_names.json",
    "feature_scaler.pkl",
    "basic_features.npz",
    "advanced_features.npz",
    "advanced_features_scaled.npz",
    "y_train.npy",
    "y_val.npy",
    "y_test.npy",
    "semantic_pair_embeddings.npz",
    "semantic_model_config.json",
    "feature_config.json",
    "feature_class_diagnostics.csv"
]

print("Feature artifacts:")
print()

for filename in expected_artifacts:

    path = FEATURE_DIR / filename

    if path.exists():

        print(
            "✓",
            filename,
            f"({path.stat().st_size / 1024:.1f} KB)"
        )

    else:

        print(
            "✗ MISSING:",
            filename
        )

Feature artifacts:

✓ word_tfidf.pkl (3898.2 KB)
✓ char_tfidf.pkl (3371.8 KB)
✓ handcrafted_feature_names.json (0.4 KB)
✓ feature_scaler.pkl (1.1 KB)
✓ basic_features.npz (10014.2 KB)
✓ advanced_features.npz (21198.5 KB)
✓ advanced_features_scaled.npz (20307.8 KB)
✓ y_train.npy (2121.4 KB)
✓ y_val.npy (595.6 KB)
✓ y_test.npy (437.5 KB)
✓ semantic_pair_embeddings.npz (1122663.1 KB)
✓ semantic_model_config.json (0.2 KB)
✓ feature_config.json (1.4 KB)
✓ feature_class_diagnostics.csv (1.1 KB)


In [57]:
# ============================================================
# CELL 51: FINAL FEATURE ENGINEERING SUMMARY
# ============================================================

print("=" * 75)
print("QUESTION SIMILARITY - FEATURE ENGINEERING COMPLETE")
print("=" * 75)

print()

print(
    "BASIC FEATURES"
)

print(
    f"  Handcrafted features : "
    f"{len(HANDCRAFTED_FEATURE_NAMES)}"
)

print(
    "  Word TF-IDF          : "
    f"{q1_word_train.shape[1]:,} dimensions"
)

print(
    "  Character TF-IDF     : "
    f"{q1_char_train.shape[1]:,} dimensions"
)

print()

print(
    "ADVANCED FEATURES"
)

print(
    f"  Semantic features    : "
    f"{len(SEMANTIC_FEATURE_NAMES)}"
)

print(
    f"  Compact advanced     : "
    f"{X_advanced_train.shape[1]} features"
)

print(
    f"  Embedding dimension  : "
    f"{unique_embeddings.shape[1]}"
)

print()

print(
    "DATA SPLITS"
)

print(
    f"  Training             : {len(y_train):,}"
)

print(
    f"  Validation           : {len(y_val):,}"
)

print(
    f"  Internal test        : {len(y_test):,}"
)

print()

print(
    "Feature artifacts saved to:"
)

print(
    FEATURE_DIR
)

print()

print(
    "✓ Feature engineering completed."
)

print(
    "✓ No model training performed."
)

QUESTION SIMILARITY - FEATURE ENGINEERING COMPLETE

BASIC FEATURES
  Handcrafted features : 15
  Word TF-IDF          : 100,000 dimensions
  Character TF-IDF     : 100,000 dimensions

ADVANCED FEATURES
  Semantic features    : 7
  Compact advanced     : 22 features
  Embedding dimension  : 384

DATA SPLITS
  Training             : 271,524
  Validation           : 76,226
  Internal test        : 55,986

Feature artifacts saved to:
C:\Users\bimal\OneDrive\Desktop\bibhu sir project\question_similarity_project\model\feature_artifacts

✓ Feature engineering completed.
✓ No model training performed.


In [58]:
# ============================================================
# TEMPORARY CELL: CHECK AVAILABLE FEATURE VARIABLES
# ============================================================

feature_variables = [
    name
    for name in globals()
    if any(
        keyword in name.lower()
        for keyword in [
            "train",
            "val",
            "test",
            "feature",
            "cosine",
            "semantic",
            "tfidf",
            "fuzzy",
            "jaccard",
            "length"
        ]
    )
]

for name in sorted(feature_variables):
    try:
        value = globals()[name]

        if hasattr(value, "shape"):
            print(
                f"{name:<40} {value.shape}"
            )

    except Exception:
        pass

X_advanced_test                          (55986, 22)
X_advanced_test_scaled                   (55986, 22)
X_advanced_train                         (271524, 22)
X_advanced_train_scaled                  (271524, 22)
X_advanced_val                           (76226, 22)
X_advanced_val_scaled                    (76226, 22)
X_basic_test                             (55986, 15)
X_basic_train                            (271524, 15)
X_basic_val                              (76226, 15)
advanced_test_df                         (55986, 22)
advanced_train_df                        (271524, 22)
advanced_val_df                          (76226, 22)
basic_test_df                            (55986, 15)
basic_train_df                           (271524, 15)
basic_val_df                             (76226, 15)
char_cos_test                            (55986,)
char_cos_train                           (271524,)
char_cos_val                             (76226,)
emb1_test                                (55986, 

In [59]:
# ============================================================
# DEBUG: FIND AVAILABLE FEATURE ARRAYS
# ============================================================

import numpy as np

print("=" * 80)
print("AVAILABLE FEATURE VARIABLES")
print("=" * 80)

for name, value in list(globals().items()):

    if name.startswith("_"):
        continue

    try:

        if isinstance(value, np.ndarray):

            if value.ndim >= 1:

                print(
                    f"{name:<40} "
                    f"shape={value.shape} "
                    f"dtype={value.dtype}"
                )

    except Exception:
        pass

AVAILABLE FEATURE VARIABLES
value                                    shape=(76226,) dtype=int64
arr                                      shape=(55986, 22) dtype=float32
y_train                                  shape=(271524,) dtype=int64
y_val                                    shape=(76226,) dtype=int64
y_test                                   shape=(55986,) dtype=int64
variable                                 shape=(55986,) dtype=int64
hand_train                               shape=(271524, 15) dtype=float32
hand_val                                 shape=(76226, 15) dtype=float32
hand_test                                shape=(55986, 15) dtype=float32
word_cos_train                           shape=(271524,) dtype=float32
word_cos_val                             shape=(76226,) dtype=float32
word_cos_test                            shape=(55986,) dtype=float32
char_cos_train                           shape=(271524,) dtype=float32
char_cos_val                             shape=(76226,) 

In [60]:
# ============================================================
# DEBUG: FIND PANDAS FEATURE OBJECTS
# ============================================================

print("=" * 80)
print("AVAILABLE DATAFRAME / SERIES VARIABLES")
print("=" * 80)

for name, value in list(globals().items()):

    if name.startswith("_"):
        continue

    try:

        if hasattr(value, "shape"):

            shape = value.shape

            if len(shape) >= 1:

                print(
                    f"{name:<40} "
                    f"shape={shape}"
                )

    except Exception:
        pass

AVAILABLE DATAFRAME / SERIES VARIABLES
arr                                      shape=(55986, 22)
train_df                                 shape=(271524, 8)
val_df                                   shape=(76226, 8)
internal_test_df                         shape=(55986, 8)
dataset                                  shape=(55986, 8)
q1_train                                 shape=(271524,)
q2_train                                 shape=(271524,)
q1_val                                   shape=(76226,)
q2_val                                   shape=(76226,)
q1_test                                  shape=(55986,)
q2_test                                  shape=(55986,)
y_train                                  shape=(271524,)
y_val                                    shape=(76226,)
y_test                                   shape=(55986,)
variable                                 shape=(55986,)
q1_word_train                            shape=(271524, 100000)
q2_word_train                            s

In [61]:
# ============================================================
# FINAL CELL: SAVE ENGINEERED FEATURES
# ============================================================

from pathlib import Path
import pickle
import numpy as np

# ------------------------------------------------------------
# 1. PROJECT PATH
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    r"C:\Users\bimal\OneDrive\Desktop\bibhu sir project"
    r"\question_similarity_project"
)

FEATURE_DIR = (
    PROJECT_ROOT
    / "model"
    / "feature_artifacts"
)

FEATURE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FEATURE_PATH = (
    FEATURE_DIR
    / "engineered_features.pkl"
)

print("Project root:")
print(PROJECT_ROOT)

print("\nFeature directory:")
print(FEATURE_DIR)

print("\nOutput file:")
print(FEATURE_PATH)


# ------------------------------------------------------------
# 2. CHECK REQUIRED VARIABLES
# ------------------------------------------------------------

required_variables = [
    "hand_train",
    "hand_val",
    "hand_test",
    "y_train",
    "y_val",
    "y_test"
]

missing = [
    name
    for name in required_variables
    if name not in globals()
]

if missing:

    print("\n❌ Missing variables:")

    for name in missing:
        print("   -", name)

    raise RuntimeError(
        "\nThe feature matrices are not available.\n"
        "Run the feature-engineering cells that create "
        "hand_train, hand_val and hand_test before "
        "running this cell."
    )


# ------------------------------------------------------------
# 3. CONVERT TO NUMPY
# ------------------------------------------------------------

hand_train = np.asarray(
    hand_train,
    dtype=np.float32
)

hand_val = np.asarray(
    hand_val,
    dtype=np.float32
)

hand_test = np.asarray(
    hand_test,
    dtype=np.float32
)

y_train = np.asarray(
    y_train,
    dtype=np.int32
)

y_val = np.asarray(
    y_val,
    dtype=np.int32
)

y_test = np.asarray(
    y_test,
    dtype=np.int32
)


# ------------------------------------------------------------
# 4. FEATURE NAMES
# ------------------------------------------------------------

if "feature_names" in globals():

    feature_names = list(
        feature_names
    )

else:

    feature_names = [
        f"feature_{i}"
        for i in range(
            hand_train.shape[1]
        )
    ]


# ------------------------------------------------------------
# 5. VALIDATION
# ------------------------------------------------------------

print("\nFeature shapes:")

print(
    "hand_train:",
    hand_train.shape
)

print(
    "hand_val:",
    hand_val.shape
)

print(
    "hand_test:",
    hand_test.shape
)

print(
    "y_train:",
    y_train.shape
)

print(
    "y_val:",
    y_val.shape
)

print(
    "y_test:",
    y_test.shape
)

print(
    "\nNumber of features:",
    hand_train.shape[1]
)


if hand_train.shape[1] != hand_val.shape[1]:

    raise RuntimeError(
        "Train and validation feature counts differ."
    )


if hand_train.shape[1] != hand_test.shape[1]:

    raise RuntimeError(
        "Train and test feature counts differ."
    )


if len(feature_names) != hand_train.shape[1]:

    raise RuntimeError(
        "Feature-name count does not match "
        "feature count."
    )


# ------------------------------------------------------------
# 6. CREATE ARTIFACT
# ------------------------------------------------------------

feature_artifact = {

    "hand_train": hand_train,

    "hand_val": hand_val,

    "hand_test": hand_test,

    "y_train": y_train,

    "y_val": y_val,

    "y_test": y_test,

    "feature_names": feature_names,

    "n_features": hand_train.shape[1]
}


# ------------------------------------------------------------
# 7. SAVE
# ------------------------------------------------------------

with open(
    FEATURE_PATH,
    "wb"
) as file:

    pickle.dump(
        feature_artifact,
        file,
        protocol=pickle.HIGHEST_PROTOCOL
    )


# ------------------------------------------------------------
# 8. VERIFY
# ------------------------------------------------------------

if not FEATURE_PATH.exists():

    raise RuntimeError(
        "File was not created."
    )


file_size_mb = (
    FEATURE_PATH.stat().st_size
    / (1024 ** 2)
)


print("\n" + "=" * 70)
print("✓ ENGINEERED FEATURES SUCCESSFULLY SAVED")
print("=" * 70)

print("\nFile:")
print(FEATURE_PATH)

print(
    "\nFile size:",
    f"{file_size_mb:.2f} MB"
)

print(
    "\nNumber of features:",
    hand_train.shape[1]
)

print("\n✓ Ready for model training.")

Project root:
C:\Users\bimal\OneDrive\Desktop\bibhu sir project\question_similarity_project

Feature directory:
C:\Users\bimal\OneDrive\Desktop\bibhu sir project\question_similarity_project\model\feature_artifacts

Output file:
C:\Users\bimal\OneDrive\Desktop\bibhu sir project\question_similarity_project\model\feature_artifacts\engineered_features.pkl

Feature shapes:
hand_train: (271524, 15)
hand_val: (76226, 15)
hand_test: (55986, 15)
y_train: (271524,)
y_val: (76226,)
y_test: (55986,)

Number of features: 15

✓ ENGINEERED FEATURES SUCCESSFULLY SAVED

File:
C:\Users\bimal\OneDrive\Desktop\bibhu sir project\question_similarity_project\model\feature_artifacts\engineered_features.pkl

File size: 24.64 MB

Number of features: 15

✓ Ready for model training.


In [62]:
# ============================================================
# VERIFY ENGINEERED FEATURES FILE
# ============================================================

from pathlib import Path
import pickle

PROJECT_ROOT = Path(
    r"C:\Users\bimal\OneDrive\Desktop\bibhu sir project"
    r"\question_similarity_project"
)

FEATURE_PATH = (
    PROJECT_ROOT
    / "model"
    / "feature_artifacts"
    / "engineered_features.pkl"
)

print("Checking:")
print(FEATURE_PATH)

if not FEATURE_PATH.exists():

    raise FileNotFoundError(
        "\n❌ engineered_features.pkl does not exist."
    )

with open(
    FEATURE_PATH,
    "rb"
) as file:

    data = pickle.load(file)


print("\n✓ File loaded successfully.")

print("\nKeys:")

for key in data:

    print(
        " ",
        key
    )

print("\nShapes:")

print(
    "hand_train:",
    data["hand_train"].shape
)

print(
    "hand_val:",
    data["hand_val"].shape
)

print(
    "hand_test:",
    data["hand_test"].shape
)

print(
    "y_train:",
    data["y_train"].shape
)

print(
    "y_val:",
    data["y_val"].shape
)

print(
    "y_test:",
    data["y_test"].shape
)

print(
    "\nNumber of features:",
    data["n_features"]
)

Checking:
C:\Users\bimal\OneDrive\Desktop\bibhu sir project\question_similarity_project\model\feature_artifacts\engineered_features.pkl

✓ File loaded successfully.

Keys:
  hand_train
  hand_val
  hand_test
  y_train
  y_val
  y_test
  feature_names
  n_features

Shapes:
hand_train: (271524, 15)
hand_val: (76226, 15)
hand_test: (55986, 15)
y_train: (271524,)
y_val: (76226,)
y_test: (55986,)

Number of features: 15


In [63]:
# ============================================================
# DIAGNOSTIC: FIND EXACT TRAINING FEATURE ORDER
# ============================================================

from pathlib import Path
import pickle
import json
import numpy as np

PROJECT_ROOT = Path(
    r"C:\Users\bimal\OneDrive\Desktop\bibhu sir project"
    r"\question_similarity_project"
)

FEATURE_DIR = (
    PROJECT_ROOT
    / "model"
    / "feature_artifacts"
)

print("=" * 70)
print("TRAINING FEATURE DIAGNOSTIC")
print("=" * 70)


# ------------------------------------------------------------
# 1. feature_names.pkl
# ------------------------------------------------------------

feature_names_path = (
    PROJECT_ROOT
    / "model"
    / "feature_names.pkl"
)

if feature_names_path.exists():

    with open(
        feature_names_path,
        "rb"
    ) as f:

        feature_names = pickle.load(f)

    print("\nfeature_names.pkl:")
    print(feature_names)

else:

    print(
        "\nfeature_names.pkl NOT FOUND"
    )


# ------------------------------------------------------------
# 2. handcrafted_feature_names.json
# ------------------------------------------------------------

handcrafted_path = (
    FEATURE_DIR
    / "handcrafted_feature_names.json"
)

if handcrafted_path.exists():

    with open(
        handcrafted_path,
        "r",
        encoding="utf-8"
    ) as f:

        handcrafted_names = json.load(f)

    print(
        "\nhandcrafted_feature_names.json:"
    )

    for i, name in enumerate(
        handcrafted_names
    ):

        print(
            i,
            "->",
            name
        )

else:

    print(
        "\nhandcrafted_feature_names.json NOT FOUND"
    )


# ------------------------------------------------------------
# 3. feature_config.json
# ------------------------------------------------------------

config_path = (
    FEATURE_DIR
    / "feature_config.json"
)

if config_path.exists():

    with open(
        config_path,
        "r",
        encoding="utf-8"
    ) as f:

        feature_config = json.load(f)

    print(
        "\nfeature_config.json:"
    )

    print(
        json.dumps(
            feature_config,
            indent=4
        )
    )

else:

    print(
        "\nfeature_config.json NOT FOUND"
    )


# ------------------------------------------------------------
# 4. advanced_features.npz
# ------------------------------------------------------------

advanced_path = (
    FEATURE_DIR
    / "advanced_features.npz"
)

if advanced_path.exists():

    data = np.load(
        advanced_path,
        allow_pickle=True
    )

    print(
        "\nadvanced_features.npz:"
    )

    print(
        "Keys:",
        data.files
    )

    for key in data.files:

        arr = data[key]

        print(
            f"{key}: "
            f"shape={arr.shape}, "
            f"dtype={arr.dtype}"
        )

else:

    print(
        "\nadvanced_features.npz NOT FOUND"
    )


# ------------------------------------------------------------
# 5. advanced_features_scaled.npz
# ------------------------------------------------------------

scaled_path = (
    FEATURE_DIR
    / "advanced_features_scaled.npz"
)

if scaled_path.exists():

    data_scaled = np.load(
        scaled_path,
        allow_pickle=True
    )

    print(
        "\nadvanced_features_scaled.npz:"
    )

    print(
        "Keys:",
        data_scaled.files
    )

    for key in data_scaled.files:

        arr = data_scaled[key]

        print(
            f"{key}: "
            f"shape={arr.shape}, "
            f"dtype={arr.dtype}"
        )

else:

    print(
        "\nadvanced_features_scaled.npz NOT FOUND"
    )


print("\n" + "=" * 70)
print("DIAGNOSTIC COMPLETE")
print("=" * 70)

TRAINING FEATURE DIAGNOSTIC

feature_names.pkl:
['feature_0', 'feature_1', 'feature_2', 'feature_3', 'feature_4', 'feature_5', 'feature_6', 'feature_7', 'feature_8', 'feature_9', 'feature_10', 'feature_11', 'feature_12', 'feature_13', 'feature_14']

handcrafted_feature_names.json:
0 -> q1_char_count
1 -> q2_char_count
2 -> q1_word_count
3 -> q2_word_count
4 -> char_length_difference
5 -> word_length_difference
6 -> char_length_ratio
7 -> word_length_ratio
8 -> common_token_count
9 -> jaccard_similarity
10 -> overlap_coefficient
11 -> common_word_ratio
12 -> exact_match
13 -> word_tfidf_cosine
14 -> char_tfidf_cosine

feature_config.json:
{
    "basic_features": [
        "q1_char_count",
        "q2_char_count",
        "q1_word_count",
        "q2_word_count",
        "char_length_difference",
        "word_length_difference",
        "char_length_ratio",
        "word_length_ratio",
        "common_token_count",
        "jaccard_similarity",
        "overlap_coefficient",
        "co

In [64]:
# ============================================================
# CELL: FIND SAVED FEATURE ARTIFACTS
# ============================================================

from pathlib import Path
import pickle
import json

PROJECT_ROOT = Path(
    r"C:\Users\bimal\OneDrive\Desktop\bibhu sir project"
    r"\question_similarity_project"
)

MODEL_DIR = PROJECT_ROOT / "model"
FEATURE_DIR = MODEL_DIR / "feature_artifacts"

print("=" * 70)
print("SAVED FEATURE ARTIFACTS")
print("=" * 70)

print("\nFeature directory:")
print(FEATURE_DIR)

if not FEATURE_DIR.exists():
    print("\n❌ Feature directory does not exist.")
else:

    print("\nFiles currently present:")

    for file in sorted(FEATURE_DIR.iterdir()):

        print(
            f" - {file.name} "
            f"({file.stat().st_size / (1024 * 1024):.2f} MB)"
        )


# ------------------------------------------------------------
# feature_names.pkl
# ------------------------------------------------------------

feature_names_path = (
    MODEL_DIR /
    "feature_names.pkl"
)

print("\n" + "-" * 70)
print("feature_names.pkl")
print("-" * 70)

if feature_names_path.exists():

    try:

        with open(
            feature_names_path,
            "rb"
        ) as f:

            names = pickle.load(f)

        print("Type:", type(names))
        print("Number:", len(names))
        print("\nNames:")

        for i, name in enumerate(names):

            print(
                f"{i}: {name}"
            )

    except Exception as e:

        print(
            "❌ Could not read feature_names.pkl"
        )

        print(e)

else:

    print(
        "❌ feature_names.pkl not found."
    )


# ------------------------------------------------------------
# handcrafted_feature_names.json
# ------------------------------------------------------------

handcrafted_path = (
    FEATURE_DIR /
    "handcrafted_feature_names.json"
)

print("\n" + "-" * 70)
print("handcrafted_feature_names.json")
print("-" * 70)

if handcrafted_path.exists():

    try:

        with open(
            handcrafted_path,
            "r",
            encoding="utf-8"
        ) as f:

            handcrafted_names = json.load(f)

        print(
            "Number:",
            len(handcrafted_names)
        )

        for i, name in enumerate(
            handcrafted_names
        ):

            print(
                f"{i}: {name}"
            )

    except Exception as e:

        print(
            "❌ Could not read handcrafted feature names."
        )

        print(e)

else:

    print(
        "❌ handcrafted_feature_names.json "
        "not found."
    )


print("\n" + "=" * 70)
print("DIAGNOSTIC FINISHED")
print("=" * 70)

SAVED FEATURE ARTIFACTS

Feature directory:
C:\Users\bimal\OneDrive\Desktop\bibhu sir project\question_similarity_project\model\feature_artifacts



Files currently present:
 - advanced_features.npz (20.70 MB)
 - advanced_features_scaled.npz (19.83 MB)
 - basic_features.npz (9.78 MB)
 - char_tfidf.pkl (3.29 MB)
 - engineered_features.pkl (24.64 MB)
 - feature_class_diagnostics.csv (0.00 MB)
 - feature_config.json (0.00 MB)
 - feature_scaler.pkl (0.00 MB)
 - handcrafted_feature_names.json (0.00 MB)
 - semantic_model_config.json (0.00 MB)
 - semantic_pair_embeddings.npz (1096.35 MB)
 - word_tfidf.pkl (3.81 MB)
 - y_test.npy (0.43 MB)
 - y_train.npy (2.07 MB)
 - y_val.npy (0.58 MB)

----------------------------------------------------------------------
feature_names.pkl
----------------------------------------------------------------------
Type: <class 'list'>
Number: 15

Names:
0: feature_0
1: feature_1
2: feature_2
3: feature_3
4: feature_4
5: feature_5
6: feature_6
7: feature_7
8: feature_8
9: feature_9
10: feature_10
11: feature_11
12: feature_12
13: feature_13
14: feature_14

-----------------------------------------------------

In [65]:
# ============================================================
# INSPECT ENGINEERED FEATURES
# ============================================================

from pathlib import Path
import joblib
import pickle
import numpy as np

PROJECT_ROOT = Path(
    r"C:\Users\bimal\OneDrive\Desktop\bibhu sir project"
    r"\question_similarity_project"
)

FEATURE_DIR = (
    PROJECT_ROOT
    / "model"
    / "feature_artifacts"
)

ENGINEERED_PATH = (
    FEATURE_DIR
    / "engineered_features.pkl"
)

print("=" * 70)
print("ENGINEERED FEATURES INSPECTION")
print("=" * 70)

print("\nFile:")
print(ENGINEERED_PATH)

if not ENGINEERED_PATH.exists():

    raise FileNotFoundError(
        f"File not found:\n{ENGINEERED_PATH}"
    )

print("\n✓ File exists.")

print(
    "File size:",
    round(
        ENGINEERED_PATH.stat().st_size /
        (1024 ** 2),
        2
    ),
    "MB"
)

# ------------------------------------------------------------
# LOAD
# ------------------------------------------------------------

try:

    data = joblib.load(
        ENGINEERED_PATH
    )

except Exception:

    with open(
        ENGINEERED_PATH,
        "rb"
    ) as f:

        data = pickle.load(f)


# ------------------------------------------------------------
# DISPLAY TYPE
# ------------------------------------------------------------

print(
    "\nObject type:",
    type(data)
)


# ------------------------------------------------------------
# DISPLAY KEYS
# ------------------------------------------------------------

if isinstance(data, dict):

    print("\nDictionary keys:")

    for key in data.keys():

        print(
            " -",
            key
        )

else:

    print(
        "\nThe artifact is not a dictionary."
    )


# ------------------------------------------------------------
# INSPECT EVERY ITEM
# ------------------------------------------------------------

if isinstance(data, dict):

    print("\n" + "-" * 70)
    print("ITEM DETAILS")
    print("-" * 70)

    for key, value in data.items():

        print(
            f"\nKEY: {key}"
        )

        print(
            "TYPE:",
            type(value)
        )

        if hasattr(
            value,
            "shape"
        ):

            print(
                "SHAPE:",
                value.shape
            )

        elif isinstance(
            value,
            list
        ):

            print(
                "LENGTH:",
                len(value)
            )

            if len(value) > 0:

                print(
                    "FIRST ITEM:",
                    value[0]
                )

        elif isinstance(
            value,
            dict
        ):

            print(
                "DICT KEYS:",
                list(value.keys())
            )

        else:

            print(
                "VALUE:",
                value
            )


print("\n" + "=" * 70)
print("INSPECTION COMPLETE")
print("=" * 70)

ENGINEERED FEATURES INSPECTION

File:
C:\Users\bimal\OneDrive\Desktop\bibhu sir project\question_similarity_project\model\feature_artifacts\engineered_features.pkl

✓ File exists.
File size: 24.64 MB

Object type: <class 'dict'>

Dictionary keys:
 - hand_train
 - hand_val
 - hand_test
 - y_train
 - y_val
 - y_test
 - feature_names
 - n_features

----------------------------------------------------------------------
ITEM DETAILS
----------------------------------------------------------------------

KEY: hand_train
TYPE: <class 'numpy.ndarray'>
SHAPE: (271524, 15)

KEY: hand_val
TYPE: <class 'numpy.ndarray'>
SHAPE: (76226, 15)

KEY: hand_test
TYPE: <class 'numpy.ndarray'>
SHAPE: (55986, 15)

KEY: y_train
TYPE: <class 'numpy.ndarray'>
SHAPE: (271524,)

KEY: y_val
TYPE: <class 'numpy.ndarray'>
SHAPE: (76226,)

KEY: y_test
TYPE: <class 'numpy.ndarray'>
SHAPE: (55986,)

KEY: feature_names
TYPE: <class 'list'>
LENGTH: 15
FIRST ITEM: feature_0

KEY: n_features
TYPE: <class 'int'>
VALUE: 15

I

In [66]:
# ============================================================
# INSPECT FEATURE CONFIGURATION
# ============================================================

import json

CONFIG_PATH = (
    FEATURE_DIR
    / "feature_config.json"
)

print("=" * 70)
print("FEATURE CONFIGURATION")
print("=" * 70)

if not CONFIG_PATH.exists():

    raise FileNotFoundError(
        f"Missing:\n{CONFIG_PATH}"
    )

with open(
    CONFIG_PATH,
    "r",
    encoding="utf-8"
) as f:

    config = json.load(f)

print(
    json.dumps(
        config,
        indent=4
    )
)

print("=" * 70)

FEATURE CONFIGURATION
{
    "basic_features": [
        "q1_char_count",
        "q2_char_count",
        "q1_word_count",
        "q2_word_count",
        "char_length_difference",
        "word_length_difference",
        "char_length_ratio",
        "word_length_ratio",
        "common_token_count",
        "jaccard_similarity",
        "overlap_coefficient",
        "common_word_ratio",
        "exact_match",
        "word_tfidf_cosine",
        "char_tfidf_cosine"
    ],
    "semantic_features": [
        "semantic_cosine",
        "semantic_euclidean",
        "semantic_manhattan",
        "semantic_abs_mean",
        "semantic_product_mean",
        "semantic_product_max",
        "semantic_product_min"
    ],
    "advanced_features": [
        "q1_char_count",
        "q2_char_count",
        "q1_word_count",
        "q2_word_count",
        "char_length_difference",
        "word_length_difference",
        "char_length_ratio",
        "word_length_ratio",
        "common_toke

In [67]:
# ============================================================
# INSPECT FEATURE NAMES
# ============================================================

FEATURE_NAMES_PATH = (
    PROJECT_ROOT
    / "model"
    / "feature_names.pkl"
)

print("=" * 70)
print("FEATURE NAMES")
print("=" * 70)

with open(
    FEATURE_NAMES_PATH,
    "rb"
) as f:

    names = pickle.load(f)

print(
    "Type:",
    type(names)
)

print(
    "Count:",
    len(names)
)

for i, name in enumerate(names):

    print(
        f"{i}: {name}"
    )

print("=" * 70)

FEATURE NAMES
Type: <class 'list'>
Count: 15
0: feature_0
1: feature_1
2: feature_2
3: feature_3
4: feature_4
5: feature_5
6: feature_6
7: feature_7
8: feature_8
9: feature_9
10: feature_10
11: feature_11
12: feature_12
13: feature_13
14: feature_14


In [68]:
# ============================================================
# CHECK TRAINING FEATURE MATRIX
# ============================================================

advanced_path = (
    FEATURE_DIR
    / "advanced_features.npz"
)

scaled_path = (
    FEATURE_DIR
    / "advanced_features_scaled.npz"
)

print("=" * 70)
print("TRAINING FEATURE MATRIX")
print("=" * 70)

advanced = np.load(
    advanced_path,
    allow_pickle=True
)

scaled = np.load(
    scaled_path,
    allow_pickle=True
)

print("\nADVANCED FEATURES")

for key in advanced.files:

    arr = advanced[key]

    print(
        key,
        "=>",
        arr.shape,
        arr.dtype
    )

print("\nSCALED ADVANCED FEATURES")

for key in scaled.files:

    arr = scaled[key]

    print(
        key,
        "=>",
        arr.shape,
        arr.dtype
    )

print("=" * 70)

TRAINING FEATURE MATRIX

ADVANCED FEATURES


X_train => (271524, 22) float32
X_val => (76226, 22) float32
X_test => (55986, 22) float32

SCALED ADVANCED FEATURES
X_train => (271524, 22) float32
X_val => (76226, 22) float32
X_test => (55986, 22) float32
